# 4 — Evaluate: ours vs baseline, one scorer

Ours vs the released baseline, on the identical val images, through the identical `pycocotools` scorer. Writes `results/results.csv` and the spot-check figure.

> Run **`1_reformat.ipynb` first, in this same VM session**. The Brackish frames live on the VM disk at `/content/data/brackish` and do not survive a restart; everything else (weights, run dirs, results) is on Drive and does.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2 · Setup — reload what 1–3 produced

In [ ]:
# Rebuild everything 1-3 left behind: the run config, the backbone, the val GT and the
# baseline metrics. Nothing is recomputed -- this notebook only reads and scores.
import torch, pathlib
from cropcounter.train import TrainConfig, build_model

os.makedirs(f'{REPO}/weights', exist_ok=True)
dst = f'{REPO}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.exists(dst):
    shutil.copy(f'{DRIVE}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth', dst)

cfg = TrainConfig.from_json('examples/FishDetection/config_8ep.json')
cfg.data_root = pathlib.Path(DATA); cfg.weights_dir = pathlib.Path('weights')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gt = json.load(open(f'{DATA}/val/annotations.json'))
baseline_metrics = json.load(open(f'{DRIVE}/results/baseline_metrics.json'))
print(f"{len(gt['images'])} val images | baselines: {list(baseline_metrics)}")

## 3 · Inference FLOPs (measured, not quoted) — ours at 1024×576

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
box_model = build_model(cfg, device); box_model.eval()
x = torch.randn(1, 3, 576, 1024, device=device)
with torch.no_grad(), FlopCounterMode(display=False) as fc:
    box_model(x)
OURS_GFLOPS = round(fc.get_total_flops() / 1e9, 1)
print(f'ours (frozen ConvNeXt-B + decoder, 1024x576): {OURS_GFLOPS} GFLOPs')
for name, m in baseline_metrics.items():
    print(f"{name}: {m.get('gflops', 'not measured -- see 3_inference')} GFLOPs")

## 4 · Results table — ours vs baseline, same images, same scorer

In [ ]:
import csv
from cropcounter.det_metrics import coco_eval, read_coco_results
from cropcounter.dinov3_pyramid import PyramidDecoder
rows = []
for run in ('brackish_frozen_s0', 'brackish_linearprobe_s0'):
    rd = f'{DRIVE}/runs/{run}'
    if not os.path.exists(f'{rd}/history.json'):
        continue
    h = json.load(open(f'{rd}/history.json'))
    best = min(range(len(h['val_loss'])), key=lambda i: h['val_loss'][i])
    row = {'model': run, 'input': '1024 long side', 'epoch': best + 1}
    for k in ('val_ap', 'val_ap50', 'val_ap75', 'val_ar100', 'val_f1', 'val_precision', 'val_recall', 'val_count_mae'):
        row[k.replace('val_', '')] = round(h[k][best], 4) if k in h else None
    # Re-score the saved predictions with the same function used for the baseline — the parity check.
    if os.path.exists(f'{rd}/predictions.json'):
        row['ap_rescored'] = round(coco_eval(f'{DATA}/val/annotations.json', read_coco_results(f'{rd}/predictions.json'))['ap'], 4)
    rows.append(row)
for name, m in baseline_metrics.items():
    rows.append({'model': f'{name} (released, likely saw these images)', 'input': name.split('_')[-1],
                 'gflops': m.get('gflops'),
                 **{k: round(m[k], 4) for k in ('ap', 'ap50', 'ap75', 'ar100')}})
# Parameter accounting — trainable AND total (the frozen ~89M backbone runs on every forward).
n_backbone = sum(p.numel() for p in box_model.backbone.parameters())
n_dec_box = sum(p.numel() for p in PyramidDecoder((128, 256, 512, 1024), task='box').parameters())
for r in rows:
    if r['model'].startswith('brackish'):
        r['trainable_params_M'] = round(n_dec_box / 1e6, 2); r['total_params_M'] = round((n_backbone + n_dec_box) / 1e6, 1)
        r['gflops'] = OURS_GFLOPS
keys = sorted({k for r in rows for k in r}, key=lambda k: (k != 'model', k))
with open(f'{DRIVE}/results/results.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
print('| ' + ' | '.join(keys) + ' |'); print('|' + '---|' * len(keys))
for r in rows:
    print('| ' + ' | '.join(str(r.get(k, '')) for k in keys) + ' |')

## 5 · Visual spot checks — GT (green) · ours (red) · RF-DETR (blue), six val images spanning the count range

In [ ]:
import numpy as np, matplotlib, matplotlib.patches
from PIL import Image
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from collections import defaultdict
ours = defaultdict(list); theirs = defaultdict(list); gts = defaultdict(list)
for d in read_coco_results(f'{DRIVE}/runs/brackish_frozen_s0/predictions.json'):
    if d['score'] >= 0.3: ours[d['image_id']].append(d['bbox'])
for d in read_coco_results(f'{DRIVE}/results/rfdetr_nano_640_predictions.json'):
    if d['score'] >= 0.3: theirs[d['image_id']].append(d['bbox'])
for a in gt['annotations']:
    gts[a['image_id']].append(a['bbox'])
name_by_id = {im['id']: os.path.basename(im['file_name']) for im in gt['images']}
by_count = sorted(name_by_id, key=lambda i: len(gts[i]))
picks = [by_count[int(q * (len(by_count) - 1))] for q in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)]
fig = Figure(figsize=(18, 12)); FigureCanvasAgg(fig)
for ax, iid in zip(fig.subplots(2, 3).ravel(), picks):
    ax.imshow(Image.open(f"{DATA}/val/images/{name_by_id[iid]}"))
    for boxes, col in ((gts[iid], 'lime'), (ours[iid], 'red'), (theirs[iid], 'deepskyblue')):
        for x, y, w, h in boxes:
            ax.add_patch(matplotlib.patches.Rectangle((x, y), w, h, fill=False, edgecolor=col, linewidth=1.2))
    ax.set_title(f"{name_by_id[iid]}  gt {len(gts[iid])} · ours {len(ours[iid])} · rfdetr {len(theirs[iid])}", fontsize=9); ax.axis('off')
fig.savefig(f'{DRIVE}/results/viz/spot_checks.png', dpi=110, bbox_inches='tight')
# Also drop the figures the report embeds into the repo checkout, to be committed from there.
DOCS_IMG = f'{REPO}/examples/FishDetection/notebooks/docs/images'
os.makedirs(DOCS_IMG, exist_ok=True)
shutil.copy(f'{DRIVE}/results/viz/spot_checks.png', f'{DOCS_IMG}/spot_checks.png')
for run in ('brackish_frozen_s0', 'brackish_linearprobe_s0'):
    if os.path.exists(f'{DRIVE}/runs/{run}/curves.png'):
        shutil.copy(f'{DRIVE}/runs/{run}/curves.png', f'{DOCS_IMG}/{run}_curves.png')
print('figures written to', DOCS_IMG)
from IPython.display import Image as IPImage, display
display(IPImage(f'{DRIVE}/results/viz/spot_checks.png'))

## 6 · Hand-off

Everything the memo needs is now on Drive: `results/results.csv`, `results/baseline_metrics.json`, `results/manifest/`, `results/throughput_probe.json`, `results/viz/spot_checks.png`, and the two run dirs. The figures the report embeds are also written into `docs/images/` in the repo checkout — commit them from there. Paste the table above into [`docs/report.md`](docs/report.md) § Results and apply the kill criterion: frozen head AP50 < ~0.5 while RF-DETR-Nano > ~0.8 ⇒ the frozen-trunk premise fails underwater → rescope to partial unfreeze or the label-efficiency claim alone.